In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("Assignment 08 - Sales Data Analysis") \
    .master("local[*]") \
    .getOrCreate()

# Display Spark information
print("SparkSession created successfully!")
print("Spark Version:", spark.version)
print("Application Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

SparkSession created successfully!
Spark Version: 4.0.5
Application Name: Assignment 08 - Sales Data Analysis
Master: local[*]


In [ ]:
import pandas as pd

# Create sample sales data
data = {
    "transaction_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_id": [201, 202, 203, 204, 205, 206, 207, 208, 209, 210],
    "product": [
        "Laptop", "Mouse", "Keyboard", "Monitor", "Laptop",
        "Mouse", "Keyboard", "Monitor", "Laptop", "Mouse"
    ],
    "amount": [55000, 500, 1500, 12000, 60000, 700, 1800, 15000, 52000, 600],
    "region": [
        "South", "North", "East", "West", "South",
        "North", "East", "West", "South", "North"
    ]
}

# Convert to Pandas DataFrame
pdf = pd.DataFrame(data)

# Save as CSV
pdf.to_csv("sales_data.csv", index=False)

print("Sales CSV created successfully!")
print(pdf.head())

Sales CSV created successfully!
   transaction_id  customer_id   product  amount region
0             101          201    Laptop   55000  South
1             102          202     Mouse     500  North
2             103          203  Keyboard    1500   East
3             104          204   Monitor   12000   West
4             105          205    Laptop   60000  South


In [ ]:
# Load CSV into Spark DataFrame
sales_df = spark.read.csv(
    "sales_data.csv",
    header=True,
    inferSchema=True
)

# Display schema
print("Sales Data Schema:")
sales_df.printSchema()

# Display records
print("Sales Data:")
sales_df.show()

Sales Data Schema:
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)

Sales Data:
+--------------+-----------+--------+------+------+
|transaction_id|customer_id| product|amount|region|
+--------------+-----------+--------+------+------+
|           101|        201|  Laptop| 55000| South|
|           102|        202|   Mouse|   500| North|
|           103|        203|Keyboard|  1500|  East|
|           104|        204| Monitor| 12000|  West|
|           105|        205|  Laptop| 60000| South|
|           106|        206|   Mouse|   700| North|
|           107|        207|Keyboard|  1800|  East|
|           108|        208| Monitor| 15000|  West|
|           109|        209|  Laptop| 52000| South|
|           110|        210|   Mouse|   600| North|
+--------------+-----------+--------+------+------+



In [ ]:
# Filter sales where amount is greater than 10000
filtered_df = sales_df.filter(sales_df.amount > 10000)

print("Filtered Sales (Amount > 10000):")

filtered_df.show()

Filtered Sales (Amount > 10000):
+--------------+-----------+-------+------+------+
|transaction_id|customer_id|product|amount|region|
+--------------+-----------+-------+------+------+
|           101|        201| Laptop| 55000| South|
|           104|        204|Monitor| 12000|  West|
|           105|        205| Laptop| 60000| South|
|           108|        208|Monitor| 15000|  West|
|           109|        209| Laptop| 52000| South|
+--------------+-----------+-------+------+------+



In [ ]:
# Count the filtered records
total_records = filtered_df.count()

print("Total sales above 10000:", total_records)

Total sales above 10000: 5


In [ ]:
from pyspark.sql.functions import sum

# Calculate total sales for each region
regional_sales = sales_df.groupBy("region").agg(
    sum("amount").alias("total_sales")
)

# Display aggregated results
print("Total Sales by Region:")

regional_sales.orderBy("region").show()

Total Sales by Region:
+------+-----------+
|region|total_sales|
+------+-----------+
|  East|       3300|
| North|       1800|
| South|     167000|
|  West|      27000|
+------+-----------+



In [ ]:
# Save aggregated results in Parquet format
regional_sales.write \
    .mode("overwrite") \
    .parquet("regional_sales_parquet")

print("Data successfully exported to Parquet!")

Data successfully exported to Parquet!


In [ ]:
# Read the saved Parquet file
parquet_df = spark.read.parquet("regional_sales_parquet")

print("Reading data from Parquet:")

parquet_df.show()

print("Parquet schema:")
parquet_df.printSchema()

Reading data from Parquet:
+------+-----------+
|region|total_sales|
+------+-----------+
| South|     167000|
|  East|       3300|
|  West|      27000|
| North|       1800|
+------+-----------+

Parquet schema:
root
 |-- region: string (nullable = true)
 |-- total_sales: long (nullable = true)

